# 윤정 추출 모델 — KoELECTRA **Base** 이진 분류 파인튜닝 (A단계)

**목표**: 가정통신문 문장을 받아 `할 일·중요 일정(1)` vs `노이즈(0)` 로 이진 분류

| 항목 | 내용 |
|------|------|
| 모델 | `monologg/koelectra-base-v3-discriminator` (Small 대비 파라미터 ~8×, 표현력 향상) |
| 입력 데이터 | `INPUT_FILE` 변수로 지정 (3번 셀에서 변경) |
| 출력 클래스 | 0: 노이즈, 1: 할 일·중요 일정 |
| GPU 권장 | **A100 / V100** — T4 는 배치 8 + gradient accumulation 으로 가능 |

**Small 과 달라진 점**

| 항목 | Small (기존) | **Base (이 노트북)** |
|------|------------|--------------------|
| Hidden size | 256 | **768** |
| 파라미터 | ~14M | **~110M** |
| 배치 크기 | 16 | **8** (+ gradient_accumulation_steps=2) |
| Mixed precision | — | **fp16=True** |
| 추론 속도 | 빠름 | 느림 (서빙 시 주의) |

**실행 방법**
1. `런타임` → `런타임 유형 변경` → **A100 GPU** (없으면 T4)
2. 3번 셀의 `INPUT_FILE` 에 학습할 JSONL 파일명 지정
3. 셀을 위에서부터 순서대로 실행 (`Shift + Enter`)
4. 마지막 셀에서 `koelectra-binary-base.zip` 다운로드 → `model/extraction/checkpoints/koelectra-binary-base/` 에 압축 풀기

## 1. 라이브러리 설치

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.0 evaluate==0.4.3 scikit-learn peft

## 2. GPU 확인

Base 모델은 GPU 메모리를 많이 씁니다. A100(40GB) 권장, T4(16GB) 는 배치 8로 가능.

In [ ]:
import torch

print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 이름   :', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU 메모리 : {total:.1f} GB')
    if total < 20:
        print('⚠️  T4(16GB) 감지 — 배치 8 + gradient_accumulation_steps=2 로 실행합니다.')
    else:
        print('✅ 충분한 메모리 — 배치 크기를 늘려도 됩니다.')

## 3. 입력 데이터 지정

`INPUT_FILE` 에 사용할 JSONL 파일명을 지정하세요.  
포맷: `{"text": str, "is_todo": bool, "is_title": bool}` (is_title 은 무시)

In [ ]:
# ── 여기서 학습 데이터 파일명을 지정하세요 ────────────────────────────────────
INPUT_FILE = 'v3.1.2_dual_labeled.jsonl'   # ← 변경 가능
# INPUT_FILE = 'v3.1_dual_labeled.jsonl'
# INPUT_FILE = 'v3.1.3_dual_labeled.jsonl'  # B그룹 제외 버전 (준비 중)
# ─────────────────────────────────────────────────────────────────────────────

print(f'입력 파일: {INPUT_FILE}')

## 4. 데이터 로드

In [ ]:
import json
from collections import Counter

with open(INPUT_FILE, encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]

texts: list[str] = []
labels: list[int] = []

for row in rows:
    text = str(row.get('text', '') or '').strip()
    if len(text) < 7:
        continue
    texts.append(text)
    labels.append(int(bool(row.get('is_todo', False))))

cnt = Counter(labels)
print(f'총 문장 수  : {len(texts)}')
print(f'라벨 분포:')
print(f'  0 (노이즈) : {cnt[0]}  ({cnt[0]/len(labels)*100:.1f}%)')
print(f'  1 (할 일)  : {cnt[1]}  ({cnt[1]/len(labels)*100:.1f}%)')

## 5. Train / Val 분할

stratified split 으로 라벨 비율을 유지합니다.

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

print(f'학습셋: {len(train_texts)}')
print(f'  양성(할 일): {sum(train_labels)}, 음성(노이즈): {len(train_labels) - sum(train_labels)}')
print(f'검증셋: {len(val_texts)}')
print(f'  양성(할 일): {sum(val_labels)}, 음성(노이즈): {len(val_labels) - sum(val_labels)}')

## 6. 토크나이저 + 데이터셋

Base 토크나이저는 Small 과 vocab 동일(35,000). `max_length=128` 로 충분합니다.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'monologg/koelectra-base-v3-discriminator'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )

train_ds = (
    Dataset.from_dict({'text': train_texts, 'label': train_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)
val_ds = (
    Dataset.from_dict({'text': val_texts, 'label': val_labels})
    .map(encode, batched=True)
    .remove_columns(['text'])
)

print('학습 데이터셋:', train_ds)
print('검증 데이터셋:', val_ds)

## 7. 모델 + 평가 지표 + 클래스 가중치

True 비율이 ~13% 로 불균형하므로 `compute_class_weight('balanced')` 로 보정합니다.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

id2label = {0: '노이즈', 1: '할 일'}
label2id = {'노이즈': 0, '할 일': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

_w = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
_class_weights = torch.tensor(_w, dtype=torch.float)
print(f'클래스 가중치: 노이즈={_w[0]:.3f}, 할 일={_w[1]:.3f}')


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fn = torch.nn.CrossEntropyLoss(weight=_class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, pos_label=1, zero_division=0),
        'precision': precision_score(labels, preds, pos_label=1, zero_division=0),
        'recall':    recall_score(labels, preds, pos_label=1, zero_division=0),
    }

## 8. 학습 실행

Base 모델 기준 학습 설정:
- **배치 8** + `gradient_accumulation_steps=2` → 유효 배치 16 (Small 과 동일)
- **fp16=True** — 혼합 정밀도로 속도 향상 및 메모리 절약
- T4 기준 약 20~30분 / A100 기준 약 8~12분

In [ ]:
from transformers import TrainingArguments, DataCollatorWithPadding

# GPU 메모리에 따라 배치 크기 자동 조정
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    BATCH_SIZE = 16 if total_mem >= 30 else 8
    GRAD_ACCUM = 1  if total_mem >= 30 else 2
else:
    BATCH_SIZE, GRAD_ACCUM = 4, 4

print(f'배치 크기: {BATCH_SIZE}, gradient_accumulation_steps: {GRAD_ACCUM}')
print(f'유효 배치: {BATCH_SIZE * GRAD_ACCUM}')

args = TrainingArguments(
    output_dir='./koelectra-binary-base-output',
    save_safetensors=False,
    num_train_epochs=10,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    fp16=torch.cuda.is_available(),          # GPU 있을 때만 fp16
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 9. 최종 평가

**할 일(1)** 클래스 F1이 핵심 지표. Small 대비 향상 여부를 확인합니다.

| 모델 | val F1 | val Recall | galsan Recall |
|------|--------|------------|---------------|
| v3.1 Small | 0.8223 | 0.8431 | 0.8455 |
| **Base (이번)** | — | — | — |

In [ ]:
from sklearn.metrics import classification_report

preds_out = trainer.predict(val_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=['노이즈', '할 일'],
    digits=4,
    zero_division=0,
))

## 10. 임계값(Threshold) 탐색

Small 에서 `BINARY_THRESHOLD=0.55` 를 사용했습니다.  
Base 는 확률 분포가 달라질 수 있으므로 0.4 ~ 0.7 구간에서 F1 최적값을 찾습니다.

In [ ]:
import torch.nn.functional as F

logits_tensor = torch.tensor(preds_out.predictions)
probs = F.softmax(logits_tensor, dim=-1)[:, 1].numpy()  # P(할 일)

print(f'{"Threshold":>10} {"F1":>8} {"Precision":>10} {"Recall":>8}')
print('-' * 42)
best_f1, best_thr = 0, 0.5
for thr in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    preds_thr = (probs >= thr).astype(int)
    f1  = f1_score(y_true, preds_thr, pos_label=1, zero_division=0)
    pre = precision_score(y_true, preds_thr, pos_label=1, zero_division=0)
    rec = recall_score(y_true, preds_thr, pos_label=1, zero_division=0)
    mark = ' ← best' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
    print(f'{thr:>10.2f} {f1:>8.4f} {pre:>10.4f} {rec:>8.4f}{mark}')

print(f'\n최적 임계값: {best_thr}  (F1={best_f1:.4f})')
print(f'predict.py 의 BINARY_THRESHOLD 를 {best_thr} 로 업데이트하세요.')

## 11. 모델 저장

In [ ]:
OUTPUT_DIR = './koelectra-binary-base'
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 최적 임계값도 함께 기록
import json
with open(f'{OUTPUT_DIR}/threshold.json', 'w') as f:
    json.dump({'BINARY_THRESHOLD': best_thr, 'val_f1': round(best_f1, 4)}, f)

print('저장 완료:', OUTPUT_DIR)
!ls -lh {OUTPUT_DIR}

## 12. 추론 테스트

저장된 Base 모델로 이진 추론을 검증합니다.

In [ ]:
from transformers import AutoModelForSequenceClassification as AM

_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_clf = AM.from_pretrained(OUTPUT_DIR, num_labels=2)
_clf.eval()

def binary_predict(sentence: str, threshold: float = best_thr) -> tuple[int, float]:
    inputs = _tok(sentence, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    with torch.no_grad():
        prob = torch.softmax(_clf(**inputs).logits, dim=-1)[0]
    label = 1 if prob[1].item() >= threshold else 0
    return label, round(float(prob[1].item()), 3)


test_sents = [
    ('학부모님 안녕하세요.',                         0),
    ('4월 30일까지 체험학습 동의서를 제출해주세요.',  1),
    ('준비물은 도시락과 물병입니다.',                 1),
    ('5월 1일은 학교자율휴업일입니다.',               1),
    ('서울갈산초등학교장',                           0),
    ('수강료 납부 기간: 매월 중순',                  1),
    ('보호자 동반하여 건강검진을 받아야 합니다.',      1),
    ('본교에서 운영 예정입니다.',                    0),
]

print(f'임계값: {best_thr}')
print(f'{"문장":<44} {"예상":>4} {"결과":>4} {"P(할일)":>8} {"OK"}')
print('-' * 72)
correct = 0
for sent, expected in test_sents:
    lbl, prob = binary_predict(sent)
    ok = lbl == expected
    correct += ok
    tag = '할 일' if lbl == 1 else '노이즈'
    exp_tag = '할 일' if expected == 1 else '노이즈'
    print(f'{sent:<44} {exp_tag:>4} {tag:>4} {prob:>8.3f}  {"✅" if ok else "❌"}')

print(f'\n정확도: {correct}/{len(test_sents)}')

## 13. 압축 + 다운로드

`koelectra-binary-base.zip` 을 `model/extraction/checkpoints/koelectra-binary-base/` 에서 풀면  
`predict.py` 의 체크포인트 경로를 `koelectra-binary-base` 로 바꿔서 사용할 수 있습니다.

In [ ]:
!zip -r koelectra-binary-base.zip koelectra-binary-base/

from google.colab import files
files.download('koelectra-binary-base.zip')

## 끝

다운로드 → 압축 풀기 → `predict.py` 체크포인트 경로 변경

### Small vs Base 성능 비교표 (채워넣기)

| 모델 | 학습 데이터 | val F1 | val Recall | galsan F1 | galsan Recall |
|------|------------|--------|------------|-----------|---------------|
| v3.1 Small | v3.1_dual_labeled (28,247) | 0.8223 | 0.8431 | 0.4168 | 0.8455 |
| Base (이번) | `INPUT_FILE` | — | — | — | — |